# 🔭 Analizar — el observatorio del corpus
Las preguntas esenciales, respondidas con la memoria acumulada:
1) ¿Que pinta tiene mi corpus? 2) ¿Quien gana donde? 3) ¿Donde duele la incertidumbre?
4) ¿El sistema aprende? 5) ¿La Fase B enmienda a la Fase A? 6) ¿Que embedding separa mejor?


In [ ]:
import sys, os, json, pathlib, re
import numpy as np
import matplotlib.pyplot as plt

# 1) API keys ANTES de importar rag_lab (CONFIG es una instantanea al importar)
# os.environ['OPENAI_API_KEY'] = '...'
# os.environ['ANTHROPIC_API_KEY'] = '...'
# os.environ['VOYAGE_API_KEY'] = '...'   # opcional
# os.environ['COHERE_API_KEY'] = '...'   # opcional

# 2) Ruta del proyecto: HARDCODEADA (en local, cambia esta linea a tu ruta)
RAIZ = pathlib.Path('/content/drive/MyDrive/rag_lab/rag_lab_proyecto')
sys.path.insert(0, str(RAIZ))

from rag_lab.config import CONFIG
from rag_lab.memory import Memoria
mem = Memoria(CONFIG.fichero_memoria)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})
print('Memoria:', CONFIG.fichero_memoria)
print('Resumen:', mem.resumen())


## 1. El mapa del corpus en el espacio de perfil
Cada punto, un documento. ¿Se forman tribus? Las tribus son donde los priors seran fuertes.


In [ ]:
ses = [s for s in mem._leer().get('sesiones', []) if s.get('perfil')]
P = [s['perfil'] for s in ses]
x = [p.get('dispersion_lda') or 0 for p in P]
y = [p.get('silueta_umap') or 0 for p in P]
c = [p.get('repetitividad_lexica') or 0 for p in P]
t = [40 + 2.5 * (p.get('n_paginas') or 10) for p in P]
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(x, y, s=t, c=c, cmap='plasma', alpha=0.75, edgecolors='k', linewidths=0.4)
for s, xi, yi in zip(ses, x, y):
    ax.annotate(str(s.get('pdf_nombre', ''))[:18], (xi, yi), fontsize=7, alpha=0.7)
plt.colorbar(sc, label='repetitividad lexica')
ax.set_xlabel('dispersion tematica (LDA)'); ax.set_ylabel('silueta semantica (UMAP)')
ax.set_title('El corpus en el espacio de perfil (tamano = paginas)')
plt.show()


## 2. ¿Quien gana donde? — la conjetura del retriever
Hipotesis acumulada: con silueta baja (espacio denso que no separa), gana lo lexico (bm25/ensemble).
Aqui la memoria responde con datos, no con intuicion.


In [ ]:
d = mem._leer()
ultimo_stack = {}
for dec in d.get('decisiones', []):
    ultimo_stack[dec['run_id']] = dec['stack']   # la ultima decision por run gana
puntos = []
for s in d.get('sesiones', []):
    st = ultimo_stack.get(s['run_id']) or {}
    for banco in ('retriever', 'embedding'):
        v = (st.get('stack_recomendado') or {}).get(banco) or {}
        if v.get('etiqueta') and s.get('perfil'):
            puntos.append({'banco': banco, 'familia': str(v['etiqueta']).split('_')[0],
                           'silueta': s['perfil'].get('silueta_umap'),
                           'lex': s['perfil'].get('repetitividad_lexica'),
                           'via': v.get('via', 'sintesis')})
ret = [p for p in puntos if p['banco'] == 'retriever' and p['silueta'] is not None]
if ret:
    fams = sorted({p['familia'] for p in ret})
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, f in enumerate(fams):
        ys = [p['silueta'] for p in ret if p['familia'] == f]
        ax.scatter(np.full(len(ys), i) + np.random.uniform(-0.08, 0.08, len(ys)), ys,
                   s=60, alpha=0.8, label=f)
    ax.set_xticks(range(len(fams))); ax.set_xticklabels(fams)
    ax.set_ylabel('silueta del documento'); ax.set_title('Familia de retriever ganadora segun la silueta')
    plt.show()
else:
    print('Aun no hay decisiones suficientes: entrena unos documentos primero.')


## 3. El termometro de la incertidumbre
¿Que bancos cuestan de resolver y por que motivo? Aqui se ve donde gastar presupuesto.


In [ ]:
from collections import Counter, defaultdict
amp = defaultdict(list); motivos = Counter(); resueltos = Counter(); total = Counter()
for st in ultimo_stack.values():
    for b, c in (st.get('certezas') or {}).items():
        a = (c.get('ic_primaria') or {}).get('amplitud')
        if a is not None: amp[b].append(a)
        total[b] += 1; resueltos[b] += int(bool(c.get('resuelto')))
        if c.get('motivo'): motivos[f"{b}:{c['motivo']}"] += 1
if amp:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
    bancos = sorted(amp)
    a1.boxplot([amp[b] for b in bancos], labels=bancos)
    a1.set_title('Amplitud del IC por banco (menos es mejor)'); a1.tick_params(axis='x', rotation=20)
    for b in bancos:
        print(f'  {b:10} resuelto {resueltos[b]}/{total[b]} veces')
    claves = motivos.most_common(10)
    a2.barh([k for k, _ in claves][::-1], [v for _, v in claves][::-1])
    a2.set_title('Motivos de disputa mas frecuentes')
    plt.tight_layout(); plt.show()
else:
    print('Sin certezas registradas todavia.')


## 4. ¿El sistema aprende? — la prueba de los priors
Cada prior emitido fue una prediccion. Aqui, su tasa de acierto acumulada: si la curva no sube
con el corpus, el aprendizaje es decorativo y hay que saberlo.


In [ ]:
preds = [p for p in mem._leer().get('predicciones', []) if p.get('acierto') is not None]
print('tasa por banco:', mem.tasa_acierto() or 'sin predicciones resueltas')
if preds:
    ac = np.cumsum([int(p['acierto']) for p in preds]) / np.arange(1, len(preds) + 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ac, marker='o', ms=3)
    ax.axhline(1/3, ls='--', alpha=0.5, label='azar aprox (3 finalistas)')
    ax.set_xlabel('prediccion #'); ax.set_ylabel('acierto acumulado')
    ax.set_ylim(0, 1); ax.legend(); ax.set_title('¿Aprende? Acierto acumulado de los priors')
    plt.show()


## 5. ¿La Fase B enmienda a la Fase A?
Cuantas coronaciones end-to-end cambiaron lo que la sintesis habia elegido. Si es frecuente,
la seleccion por componentes aislados se equivoca a menudo y la final factorial esta pagando su sitio.


In [ ]:
con_fase_b = [st for st in ultimo_stack.values() if st.get('fase_b')]
enmiendas = Counter()
for st in con_fase_b:
    for b, v in (st.get('stack_recomendado') or {}).items():
        if v.get('via') == 'fase_b':
            enmiendas[b] += 1
print(f'corridas con Fase B: {len(con_fase_b)} de {len(ultimo_stack)}')
print('bancos enmendados por la final (via=fase_b):', dict(enmiendas) or 'ninguno aun')


## 6. ¿Que embedding separa mejor ESTE documento?
Diagnostico humano (los scatter no van a ningun LLM): mismo troceado, distinta proyeccion.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from rag_lab.analisis.umap_comparativo import comparar_embeddings_umap, graficar
docs = PyPDFLoader(str(RAIZ.parent / 'pdfs' / 'MI_DOCUMENTO.pdf')).load()  # ajusta
res = comparar_embeddings_umap(docs)  # o claves=['bge-m3', 'voyage-3-lite']
fig = graficar(res)
